# Foundations capstone

This notebook is the proof that you have mastered Foundations. It has two parts:

- **Part A, a tabular project end to end:** predict whether a person earns more than \$50K a year from
  the 1994 US census ("Adult") data. You explore the data, guard against leakage, compare a baseline, a
  logistic regression and gradient boosting with cross-validation, choose a decision threshold, evaluate
  **once** on the test set with a bootstrap confidence interval, audit the model per group, and write a
  model card. It runs on any laptop in under 10 minutes.
- **Part B, an MLP on MNIST in PyTorch:** run at least three controlled experiments, plot learning curves,
  and reach **at least 98% test accuracy**. Part B is fastest on a Colab GPU and also runs on a laptop CPU.
  In [Google Colab](https://colab.research.google.com/) choose Runtime > Change runtime type > T4 GPU;
  PyTorch is preinstalled there.

Every task has `TODO` cells and a check cell that prints `OK`. The checks are light: they catch common
mistakes, not every mistake. Passing them is necessary, not sufficient. Your write-up and model card are
graded too (see the rubric on the capstone page).

**Rules of the game.** Decide everything (features, model, threshold) using the training data and
cross-validation only. The test set is opened exactly once per part, through the helper functions given
below. If you need to start again, restart the kernel and run all cells.

# Part A: income prediction on the Adult census data

## A0. Setup and data (given)

The data come from [OpenML](https://www.openml.org/d/1590) through scikit-learn, so no account is needed
(the first download is a few MB, then it is cached). There are 48,842 people, 14 feature columns and the label
`class` (`>50K` or `<=50K`). We turn the label into `income_gt_50k` (1 or 0). The given code reserves a
stratified 20% holdout now. From here, `df` contains only training rows. Leave `_holdout` untouched until A6.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

TARGET = "income_gt_50k"

def load_adult_splits():
    frame = fetch_openml("adult", version=2, as_frame=True).frame.copy()
    frame[TARGET] = (frame.pop("class") == ">50K").astype(int)
    return train_test_split(frame, test_size=0.2, stratify=frame[TARGET], random_state=0)

df, _holdout = load_adult_splits()
print(len(df), "train rows,", len(_holdout), "reserved test rows")
df.head()

## A1. Explore

Compute:

- `pos_rate`: the share of people with `income_gt_50k == 1`;
- `missing`: a Series with the number of missing values per column;
- `rate_by_sex` and `rate_by_education`: the positive rate per value of `sex` and of `education-num`
  (use `groupby(..., observed=True)`).

Then make a figure with at least two labelled panels that tell you something useful, for example the
positive rate by `education-num`, and the distribution of `age` or `hours-per-week` for each class.
Write two or three sentences on what you see, including how imbalanced the label is and what that means
for accuracy as a metric.

In [ ]:
pos_rate = None           # TODO
missing = None            # TODO
rate_by_sex = None        # TODO
rate_by_education = None  # TODO

# TODO: your figure

In [ ]:
assert pos_rate is not None and abs(pos_rate - df[TARGET].mean()) < 1e-9, "pos_rate is the mean of the 0/1 label"
assert missing is not None and int(missing["occupation"]) == int(df["occupation"].isna().sum()), "missing: df.isna().sum()"
assert rate_by_sex is not None and abs(rate_by_sex["Female"] - df.loc[df["sex"] == "Female", TARGET].mean()) < 1e-9, "rate_by_sex: groupby sex, mean of the label"
assert rate_by_education is not None and len(rate_by_education) == 16, "rate_by_education: one rate per value of education-num"
print(f"positive rate {pos_rate:.3f}: always predicting 0 is already {1 - pos_rate:.1%} accurate")
print("OK")

*Your notes on the exploration:* ...

## A2. Leakage scan and feature choices

Before modelling, check each column on its own. For every column of `df` except the target, compute the
mean 3-fold cross-validated ROC AUC of a one-feature model and store them in a Series `single_auc`
(index: column name). Use a `StandardScaler` + `LogisticRegression` pipeline for numeric columns and a
`SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore")` +
`LogisticRegression(max_iter=1000)` pipeline for categorical ones. A single column with an AUC near 1 is
almost always leakage (information that would not exist when the prediction is made).

Then build `X` (features) and `y` (the label as a NumPy array). Decide which columns to use, and write
down why. You must at least:

- leave out the target;
- leave out `fnlwgt`: it is a weight the Census Bureau assigns afterwards to make the sample represent
  the population, not a property of the person you could know when making a prediction;
- keep only one of `education` and `education-num` (they hold the same information: check it).

Also decide whether `sex` and `race` should be model inputs. Either way, you will need `sex` for the
audit in A7, so it stays in `df`. Justify your decision in the notes cell (the F11 lessons on proxies are
relevant: look at `relationship`).

In [ ]:
single_auc = None  # TODO: pd.Series, one AUC per column of df except the target
X = None           # TODO
y = None           # TODO

In [ ]:
assert single_auc is not None and set(single_auc.index) == set(df.columns) - {TARGET}, "single_auc needs one entry per column except the target"
assert single_auc.between(0.4, 1.0).all(), "AUCs should lie between about 0.5 and 1"
assert X is not None and TARGET not in X.columns, "X must not contain the target"
assert "fnlwgt" not in X.columns, "leave out fnlwgt (a sampling weight, not a property of the person)"
assert not {"education", "education-num"} <= set(X.columns), "keep only one of education and education-num"
assert single_auc[list(X.columns)].max() < 0.95, "a feature that predicts the label almost perfectly on its own is probably leakage"
assert y is not None and len(y) == len(X) == len(df) and set(np.unique(y)) == {0, 1}, "y: the 0/1 label as an array, one per row"
print(f"{X.shape[1]} features; strongest single feature: {single_auc[list(X.columns)].idxmax()} (AUC {single_auc[list(X.columns)].max():.3f})")
print("OK")

*Your feature decisions and why (including `sex` and `race`):* ...

## A3. Prepare the final evaluation (given)

The test rows were reserved before A1. The helper applies the feature choices you made using the
training rows and returns the probabilities, labels and groups in matching order. Call it only in A6,
after choosing the model and threshold. It refuses a second call. If you need to re-run, restart the
kernel and run all cells.

In [ ]:
X_train, y_train = X, y
_test_columns = list(X_train.columns)

_test_calls = 0
def predict_test_once(model):
    "Return aligned test probabilities, labels and groups once per kernel session."
    global _test_calls
    _test_calls += 1
    assert _test_calls == 1, "The test set has already been used. Make decisions with CV, then restart and run all."
    probabilities = model.predict_proba(_holdout[_test_columns])[:, 1]
    return probabilities, _holdout[TARGET].to_numpy(), _holdout["sex"].to_numpy()

## A4. Leakage-safe pipelines and model comparison with CV

Every preprocessing step (imputation, scaling, one-hot encoding) must be **inside** a `Pipeline`, so it is
re-fitted on each training fold and never sees the validation fold.

1. Build `models`, a dict with three entries:
   - `"dummy"`: `DummyClassifier(strategy="prior")`;
   - `"logistic"`: a `Pipeline` of a `ColumnTransformer` (numeric: median imputation then scaling;
     categorical: most-frequent imputation then `OneHotEncoder(handle_unknown="ignore")`) and
     `LogisticRegression(max_iter=2000)`;
   - `"boosting"`: gradient boosting. `HistGradientBoostingClassifier(categorical_features="from_dtype",
     random_state=0)` handles the pandas categorical columns and missing values itself (F8, lesson on
     gradient boosting), so it needs no preprocessing.
2. Fill `results`, a dict `name: array of the 5 fold scores` from `cross_val_score(model, X_train,
   y_train, cv=cv, scoring=...)`. Choose your scoring metric and justify it in the notes cell (ROC AUC
   is a sensible default for a ranking model on imbalanced data; average precision is another).
3. Optional: tune one or two boosting hyperparameters (`learning_rate`, `max_leaf_nodes`, `max_iter`)
   with `GridSearchCV` on the same `cv`. Keep the grid small.

Print a table of mean and standard deviation per model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
SCORING = "roc_auc"   # change if you justify another metric
models = {}   # TODO
results = {}  # TODO

In [ ]:
assert set(results) >= {"dummy", "logistic", "boosting"}, f"results needs dummy, logistic and boosting, has {sorted(results)}"
assert all(np.shape(scores) == (5,) for scores in results.values()), "each model needs five fold scores"
assert isinstance(models["logistic"], Pipeline), "the logistic model must be a Pipeline so preprocessing happens inside each fold"
for name, scores in results.items():
    expected = cross_val_score(models[name], X_train, y_train, cv=cv, scoring=SCORING)
    np.testing.assert_allclose(scores, expected, err_msg=f"{name}: use the stated metric and CV folds")
_m = {n: np.mean(s) for n, s in results.items()}
print({n: round(v, 4) for n, v in _m.items()})
print("OK")

*Which model wins, by how much compared with the fold-to-fold spread, and why your metric:* ...

## A5. Choose the decision threshold from out-of-fold predictions

Set `best_name` to your chosen model and `best_model = models[best_name]` (or your tuned estimator).
Compute `oof`, the out-of-fold probabilities of class 1 on the training set with `cross_val_predict(...,
cv=cv, method="predict_proba")[:, 1]`.

Now pick a decision rule and write it down **before** looking at the test set. Imagine the model selects
people for a mailing about a financial product aimed at higher earners: a letter costs little, but a
mailing that mostly reaches the wrong people is wasted. Some reasonable rules:

- the threshold that maximises F1 on `oof`;
- the highest threshold with recall at least some target (as in the F7 homework);
- the threshold that minimises an expected cost you state, for example a missed high earner costs 3 times
  a wasted letter.

Store the number in `threshold` and the rule in one sentence in `threshold_rule`. Plot precision and
recall against the threshold, and mark your choice.

In [ ]:
best_name = None       # TODO
best_model = None      # TODO
oof = None             # TODO
threshold = None       # TODO
threshold_rule = ""    # TODO: one sentence

In [ ]:
assert oof is not None and np.shape(oof) == (len(y_train),), "oof: one out-of-fold probability per training row"
_expected_oof = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
np.testing.assert_allclose(oof, _expected_oof, err_msg="Predict each training row with a model that did not fit that row")
_oof_auc = roc_auc_score(y_train, oof)
assert threshold is not None and 0 < threshold < 1, "threshold should be a probability strictly between 0 and 1"
assert len(threshold_rule.split()) >= 5, "write the rule as a sentence"
assert _test_calls == 0, "the threshold must be chosen without the test set"
print(f"OOF AUC {_oof_auc:.4f}, threshold {threshold:.3f}")
print("OK")

## A6. Evaluate once on the test set, with a bootstrap confidence interval

1. Fit `best_model` on all of `X_train, y_train`, then call
   `test_proba, y_test, sex_test = predict_test_once(best_model)`
   (once!) and `test_pred = (test_proba >= threshold).astype(int)`.
2. Compute `test_auc` (ROC AUC of `test_proba`) and `test_metric`, the metric your threshold rule
   targets (for example F1 of `test_pred`).
3. Write `bootstrap_ci(y_true, y_score, metric, n_boot=1000, seed=0)`: resample the test rows with
   replacement `n_boot` times, recompute `metric(y_true[idx], y_score[idx])` each time, and return the
   2.5th and 97.5th percentiles as a tuple (F5, estimation and the bootstrap). Use it for `ci_auc` and
   `ci_metric`.

Report each number with its interval. Is the test score consistent with the CV estimate?

In [ ]:
def bootstrap_ci(y_true, y_score, metric, n_boot=1000, seed=0):
    # TODO
    pass

test_proba = None   # TODO
test_pred = None    # TODO
test_auc = None     # TODO
test_metric = None  # TODO
ci_auc = None       # TODO
ci_metric = None    # TODO

In [ ]:
_lo, _hi = bootstrap_ci(np.array([0, 1, 1, 0, 1, 0, 1, 1]), np.array([0, 1, 0, 0, 1, 1, 1, 1]), accuracy_score, n_boot=500)
assert _lo is not None and 0 <= _lo < 0.75 < _hi <= 1, "bootstrap_ci should return (low, high) around the accuracy 0.75 of this toy example"
assert bootstrap_ci(np.ones(20), np.ones(20), accuracy_score, n_boot=50) == (1.0, 1.0), "a perfect classifier has a CI of (1, 1)"
assert _test_calls == 1, "call predict_test_once exactly once"
assert test_auc is not None and abs(test_auc - roc_auc_score(y_test, test_proba)) < 1e-9, "test_auc: roc_auc_score(y_test, test_proba)"
assert np.array_equal(np.asarray(test_pred), (test_proba >= threshold).astype(int)), "test_pred: test_proba >= threshold"
assert 0 <= ci_auc[0] <= ci_auc[1] <= 1, "ci_auc must be an ordered pair of AUC bounds"
assert np.isfinite(ci_metric).all() and ci_metric[0] <= ci_metric[1], "ci_metric must have finite, ordered bounds"
print("OK")

## A7. Fairness check by group

Write `group_table(y_true, y_pred, groups)` returning a DataFrame with one row per group (index: the
group value) and columns `n`, `base_rate`, `selection_rate`, `tpr`, `fpr`, `precision` (F11, measuring
fairness). Apply it to the test set by `sex` and store the result in `audit`. Compute
`dp_gap` (difference in selection rates) and `eo_gap` (difference in TPRs) between the groups, as
non-negative numbers.

In the notes cell, answer: which gaps are large, and how much of the selection-rate gap is explained by
different base rates? If you left `sex` out of the features, why does the model still treat the groups
differently? (Look at `relationship`.) Would you use a per-group threshold here? Optionally repeat the
audit by `race`, and note how small some groups are.

In [ ]:
def group_table(y_true, y_pred, groups):
    # TODO
    pass

audit = None   # TODO
dp_gap = None  # TODO
eo_gap = None  # TODO

In [ ]:
assert audit is not None and sorted(audit.index) == ["Female", "Male"], "audit: one row per sex"
assert {"n", "base_rate", "selection_rate", "tpr", "fpr", "precision"} <= set(audit.columns), "audit is missing a column"
_f = (np.asarray(sex_test) == "Female"); _p = np.asarray(test_pred); _y = np.asarray(y_test)
assert int(audit.loc["Female", "n"]) == int(_f.sum()), "n: number of test rows in the group"
assert abs(audit.loc["Female", "selection_rate"] - _p[_f].mean()) < 1e-9, "selection_rate: mean of the predictions in the group"
assert abs(audit.loc["Male", "tpr"] - _p[~_f & (_y == 1)].mean()) < 1e-9, "tpr: share of true positives in the group that are predicted 1"
assert abs(audit.loc["Female", "fpr"] - _p[_f & (_y == 0)].mean()) < 1e-9, "fpr: share of actual negatives in the group that are predicted 1"
assert abs(dp_gap - abs(audit["selection_rate"].diff().iloc[-1])) < 1e-9 and abs(eo_gap - abs(audit["tpr"].diff().iloc[-1])) < 1e-9, "gaps: absolute differences between the two rows"
print("OK")

*Your fairness findings:* ...

## A8. Model card (one page)

Fill in this card with your own numbers. Keep it to about one page: someone deciding whether to use the
model should be able to read it in three minutes.

**Model details.** Type, library and version, date, author, what the features are.

**Intended use.** What decisions it supports. **Out of scope:** what it must not be used for (for
example decisions about individuals' credit, jobs or housing), and why.

**Data.** Source (1994 US Census via OpenML), size, the time and place it describes, known gaps (missing
values, under-represented groups), and what you removed.

**Metrics.** The metric you chose and why; CV mean and spread; the test result with its 95% bootstrap
interval; the threshold and the rule behind it.

**Quantitative results by group.** Your `audit` table (by `sex`, optionally `race`), with the gaps.

**Ethical considerations, caveats and recommendations.** Proxies, the age of the data, the threshold
trade-off, and what you would check before any real deployment.

## A9. Optional: a Kaggle submission

If you have a Kaggle account, apply the same workflow to a tabular competition in Kaggle's
[Playground Series](https://www.kaggle.com/competitions?listOption=active&searchQuery=playground) (monthly
beginner-friendly contests). Compare your local CV score with the public leaderboard score: are they
consistent? This is optional and not graded.

# Part B: an MLP on MNIST in PyTorch (Colab)

Run Part B in Colab with a GPU (Runtime > Change runtime type > T4 GPU). It also runs on a CPU, more
slowly. PyTorch and torchvision are preinstalled on Colab; locally, install them first (for example
`uv run --with jupyter --with torch --with torchvision jupyter lab`).

## B0. Setup and data (given)

MNIST is downloaded once (about 12 MB) and held in memory as tensors, normalised with the usual mean and
standard deviation. 5,000 training images are held out as a **validation set**: you compare experiments on
validation accuracy. The test set is opened once, at the end, through `test_accuracy_once`.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def to_tensors(ds):
    x = ds.data.float().div(255).sub(0.1307).div(0.3081)   # (n, 28, 28)
    return x.to(device), ds.targets.to(device)

Xtr_all, ytr_all = to_tensors(datasets.MNIST("data", train=True, download=True))
Xte, yte = to_tensors(datasets.MNIST("data", train=False, download=True))
perm = torch.randperm(len(Xtr_all), generator=torch.Generator().manual_seed(0)).to(device)
Xtr, ytr = Xtr_all[perm[:55000]], ytr_all[perm[:55000]]
Xval, yval = Xtr_all[perm[55000:]], ytr_all[perm[55000:]]
print(Xtr.shape, Xval.shape, Xte.shape)

@torch.no_grad()
def evaluate(model, X, y, batch_size=5000):
    "Mean cross-entropy loss and accuracy of model on (X, y), in eval mode."
    model.eval()
    loss = correct = 0.0
    for i in range(0, len(X), batch_size):
        logits = model(X[i:i + batch_size])
        loss += F.cross_entropy(logits, y[i:i + batch_size], reduction="sum").item()
        correct += (logits.argmax(1) == y[i:i + batch_size]).sum().item()
    return loss / len(X), correct / len(X)

_test_calls_b = 0
def test_accuracy_once(model):
    "Test accuracy of model. Allowed once per kernel session."
    global _test_calls_b
    _test_calls_b += 1
    assert _test_calls_b == 1, "The MNIST test set has already been used. Restart and run all."
    return evaluate(model, Xte, yte)[1]

## B1. A configurable MLP

Write `make_mlp(sizes, dropout=0.0)` that returns an `nn.Sequential`: `nn.Flatten()`, then for each
hidden layer `nn.Linear`, `nn.ReLU()` and (if `dropout > 0`) `nn.Dropout(dropout)`, and a final
`nn.Linear` to the 10 logits with no activation. For example `sizes=[784, 256, 10]` has one hidden layer
of 256 units.

In [ ]:
def make_mlp(sizes, dropout=0.0):
    # TODO
    pass

In [ ]:
_m = make_mlp([784, 256, 10])
assert _m is not None and _m(torch.zeros(3, 28, 28)).shape == (3, 10), "make_mlp([784, 256, 10]) should map (3, 28, 28) to (3, 10) logits"
assert sum(p.numel() for p in _m.parameters()) == 784 * 256 + 256 + 256 * 10 + 10, "wrong number of parameters for [784, 256, 10]"
_m2 = make_mlp([784, 512, 256, 10], dropout=0.2)
assert sum(isinstance(l, nn.Linear) for l in _m2) == 3 and sum(isinstance(l, nn.Dropout) for l in _m2) == 2, "two hidden layers: three Linear and two Dropout"
assert not isinstance(list(_m2)[-1], (nn.ReLU, nn.Dropout)), "the last layer must output raw logits"
print("OK")

## B2. A training function

Write `train(config)` that seeds everything with `torch.manual_seed(config["seed"])`, builds
`make_mlp(config["sizes"], config["dropout"])` on `device`, creates the optimiser named by
`config["optimiser"]` (`"adam"` or `"sgd"`; use `momentum=0.9` for SGD) with `lr=config["lr"]` and
`weight_decay=config["weight_decay"]`, and trains for `config["epochs"]` epochs on mini-batches of
`config["batch_size"]` from a fresh random permutation of `Xtr` each epoch (F9: the training loop).

After every epoch, append to `history` a dict with `epoch`, `train_loss` (the mean training loss over
the epoch's batches), and `val_loss`, `val_acc` from `evaluate(model, Xval, yval)`. Remember
`model.train()` before training batches (`evaluate` switches to eval mode). Return `model, history`.

In [ ]:
baseline = dict(sizes=[784, 256, 10], dropout=0.0, optimiser="adam", lr=1e-3, weight_decay=0.0,
                epochs=10, batch_size=128, seed=0)

def train(config):
    # TODO
    pass

In [ ]:
_t = time.time()
_model, _hist = train({**baseline, "epochs": 1})
assert _hist is not None and len(_hist) == 1 and {"epoch", "train_loss", "val_loss", "val_acc"} <= set(_hist[0]), "history: one dict per epoch with epoch, train_loss, val_loss, val_acc"
assert _hist[0]["val_acc"] > 0.93, f"one epoch of the baseline should reach over 93% validation accuracy, got {_hist[0]['val_acc']:.3f}"
_model2, _hist2 = train({**baseline, "epochs": 1})
assert abs(_hist[0]["val_loss"] - _hist2[0]["val_loss"]) < 1e-3, "two runs with the same seed should match: seed with torch.manual_seed(config['seed']) first"
print(f"one epoch: val acc {_hist[0]['val_acc']:.4f} in {(time.time() - _t) / 2:.1f} s")
print("OK")

## B3. Experiment log: at least three controlled changes

Run the baseline, then at least three experiments. Each experiment changes **one kind of thing**
compared with the run it builds on, so you can attribute the difference:

- **architecture:** `sizes` (width or depth);
- **optimisation:** `optimiser` and/or `lr`;
- **regularisation:** `dropout` and/or `weight_decay`.

Cover all three kinds. Keep `seed`, `epochs` and `batch_size` fixed so runs are comparable (you may
change `epochs` for a final run). Append each run to `log` as a dict with `name`, `config`, `parent`
(the index in `log` of the run it changes, `None` for the baseline), `history`, `model` and a one-line
`note` with what you expected and what happened. Choose using **validation** accuracy, never the test
set.

On a T4 each 10-epoch run takes well under a minute; on a CPU allow a few minutes each.

In [ ]:
def changed_kinds(a, b):
    "Which kinds of setting differ between configs a and b."
    kinds = {"architecture": ["sizes"], "optimisation": ["optimiser", "lr"], "regularisation": ["dropout", "weight_decay"],
             "other": ["epochs", "batch_size", "seed"]}
    return {k for k, keys in kinds.items() if any(a[key] != b[key] for key in keys)}

log = []
def run(name, config, parent, note=""):
    t = time.time()
    model, history = train(config)
    log.append(dict(name=name, config=config, parent=parent, history=history, model=model, note=note))
    print(f"[{len(log) - 1}] {name}: val acc {history[-1]['val_acc']:.4f} (best {max(h['val_acc'] for h in history):.4f}), {time.time() - t:.0f} s")

run("baseline", baseline, None)
# TODO: at least three more runs, for example
# run("wider", {**baseline, "sizes": [784, 512, 10]}, parent=0, note="...")

In [ ]:
assert len(log) >= 4, "log needs the baseline plus at least three experiments"
assert log[0]["parent"] is None and log[0]["config"] == baseline, "log[0] should be the baseline run"
_kinds = set()
for i, r in enumerate(log[1:], start=1):
    assert r["parent"] is not None and 0 <= r["parent"] < i, f"run {i}: parent should be the index of an earlier run"
    ch = changed_kinds(log[r["parent"]]["config"], r["config"])
    assert len(ch) == 1 and "other" not in ch, f"run {i} ({r['name']}) changes {ch or 'nothing'}: change exactly one kind of setting"
    assert len(r["history"]) == r["config"]["epochs"], f"run {i}: history should have one entry per epoch"
    _kinds |= ch
assert _kinds == {"architecture", "optimisation", "regularisation"}, f"cover all three kinds of change; so far {_kinds}"
print(pd.DataFrame([dict(run=i, name=r["name"], parent=r["parent"], best_val_acc=max(h["val_acc"] for h in r["history"]),
                         final_val_acc=r["history"][-1]["val_acc"], final_train_loss=r["history"][-1]["train_loss"],
                         final_val_loss=r["history"][-1]["val_loss"]) for i, r in enumerate(log)]).round(4))
print("OK")

## B4. Learning curves

Plot, for each run in `log`, the training loss and the validation loss against the epoch (left panel)
and the validation accuracy against the epoch (right panel). Use one colour per run, solid lines for
training and dashed for validation, labelled axes, a legend and a title. Which runs overfit, and from
which epoch? Did each change do what you expected?

In [ ]:
# TODO: learning-curve figure

*What the curves show:* ...

## B5. Final model and the test set (once)

Pick `final_model`: either the best run in `log` by validation accuracy, or a new run that combines the
changes that helped (add it to `log` with `run(...)` first, choosing it on validation accuracy). Then call
`test_acc = test_accuracy_once(final_model)` once and print it. The target is **at least 0.98**. If you
fall short, go back to B3, not to the test set: restart, improve on validation, and run all again.
The target is close to what a plain MLP can do, so expect to need a combination: in the reference
solution two hidden layers (512 and 256 units) with dropout 0.2, trained for 15 epochs, reached about
98.1% validation and 98.2% test accuracy.

In [ ]:
final_model = None  # TODO
test_acc = None     # TODO
print(f"MNIST test accuracy: {test_acc}")

In [ ]:
assert _test_calls_b == 1, "call test_accuracy_once exactly once"
assert test_acc is not None and test_acc >= 0.98, f"test accuracy {test_acc} is below 0.98: improve on validation and run all again"
print("OK")

# Write-up

Replace this cell with a short report (about one page in total, plus the model card in A8):

1. **Part A:** the problem, your features and why, the CV comparison (mean and spread), the threshold
   rule, the test result with its bootstrap interval next to the CV estimate, and the fairness findings
   in two or three sentences.
2. **Part B:** the experiment log as a small table (change, expectation, validation result), what the
   learning curves showed, and the final test accuracy.
3. **What you would do next** with one more week, and one thing that surprised you.

Before submitting: Runtime > Restart and run all (or Kernel > Restart & Run All) and check that every
check cell prints `OK`.